# 02. Classification Accuracy Validation

Validate the accuracy of yolov8_animeface 3-way classification results.

| Validation Item | Method |
|----------|------|
| Default PFP | Reused from pre-filter — accuracy check |
| Anime PFP | YOLOv8 detection results vs manual labels |
| Conf tuning | F1 comparison at various thresholds |

**Input**: `data/processed/classified_3cat.csv`, manually labeled samples  
**Output**: Confusion Matrix, F1-score, optimal conf threshold

In [ ]:
import json
import random
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

DATA_DIR = Path('../data')
PROC_DIR = DATA_DIR / 'processed'
AVATAR_DIR = DATA_DIR / 'raw' / 'avatars'

df = pd.read_csv(PROC_DIR / 'classified_3cat.csv')
print(f'Total users: {len(df)}')
print(f'PFP type distribution:\n{df["profile_type"].value_counts()}')

## 1. Extract Manual Labeling Samples

Uniformly sample from each category for manual labeling.  
**Run this cell only once** — afterward, edit `manual_labels.csv` directly to input labels.

In [ ]:
SAMPLE_PATH = PROC_DIR / 'manual_labels.csv'

if not SAMPLE_PATH.exists():
    random.seed(42)
    samples = []
    for ptype in ['Anime', 'Default', 'Photo']:
        group = df[df['profile_type'] == ptype]
        n = min(100, len(group))
        sampled = group.sample(n=n, random_state=42)
        samples.append(sampled)

    sample_df = pd.concat(samples)[['uid', 'profile_type', 'anime_conf', 'clip_anime_score', 'clip_human_score']].copy()
    sample_df = sample_df.sort_values(by='uid', ascending=True, key=lambda x: x.astype(int))
    sample_df = sample_df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle
    sample_df['manual_label'] = ''  # column to fill manually
    sample_df.to_csv(SAMPLE_PATH, index=False, encoding='utf-8-sig')
    print(f'Manual labeling sample created: {len(sample_df)} images → {SAMPLE_PATH}')
    print('Open this file and enter Anime/Default/Photo in the manual_label column.')
else:
    sample_df = pd.read_csv(SAMPLE_PATH)
    labeled = sample_df[sample_df['manual_label'].notna() & (sample_df['manual_label'] != '')]
    print(f'Manual labeling progress: {len(labeled)}/{len(sample_df)} images done')


## 2. Sample Image Preview (labeling aid)

In [ ]:
# Preview top 20 anime-classified samples
anime_samples = sample_df.copy()
# anime_samples를 한 줄에 5개씩 모두 출력
n_cols = 5
chunks = [anime_samples.iloc[i:i + n_cols] for i in range(0, len(anime_samples), n_cols)]

fig, axes = plt.subplots(len(chunks), n_cols, figsize=(n_cols * 2, len(chunks) * 2))
if len(chunks) == 1:
    axes = [axes]  # Make it iterable for just 1 row

for row_axes, chunk in zip(axes, chunks):
    for ax, (_, row) in zip(row_axes, chunk.iterrows()):
        img_path = AVATAR_DIR / f"{row['uid']}.png"
        if img_path.exists():
            img = Image.open(img_path)
            ax.imshow(img)
        ax.set_title(f"{row['uid']} {row['profile_type']}", fontsize=8)
        ax.axis('off')
    # If less than n_cols in the chunk, hide the rest
    for ax in row_axes[len(chunk):]:
        ax.axis('off')

plt.suptitle('Labeling (5 per row)', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Accuracy Evaluation (run after manual labeling)

In [ ]:
DONE_MANUAL_LABELING = False

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

sample_df = pd.read_csv(SAMPLE_PATH)
labeled = sample_df[sample_df['manual_label'].notna() & (sample_df['manual_label'] != '')].copy()

if not DONE_MANUAL_LABELING:
    print(f'Please fill manual_labels.csv first.')
else:
    # If manual_label is blank, use profile_type as the manual_label for evaluation
    labeled['manual_label'] = labeled.apply(
        lambda row: row['manual_label'] if pd.notna(row['manual_label']) and str(row['manual_label']).strip() != '' else row['profile_type'],
        axis=1
    )
    
    y_true = labeled['manual_label']
    y_pred = labeled['profile_type']
    
    labels = ['Anime', 'Default', 'Photo']
    print('=== Classification Report ===')
    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(cm, display_labels=labels)
    disp.plot(ax=ax, cmap='Blues')
    ax.set_title('Confusion Matrix (yolov8 3-way classification)', fontsize=14)
    plt.tight_layout()
    plt.savefig("../figures/confusion_matrix.png", dpi=150)
    plt.show()

## 4. Confidence Threshold Tuning

In [ ]:
from sklearn.metrics import f1_score

if DONE_MANUAL_LABELING:
    # compute F1 at various conf thresholds
    # only where anime_conf exists in labeled
    labeled_with_conf = labeled[labeled['anime_conf'].notna()].copy()
    # If manual_label is empty or NaN, fill it with profile_type for this evaluation
    labeled_with_conf['manual_label'] = labeled_with_conf.apply(
        lambda row: row['manual_label'] if pd.notna(row['manual_label']) and str(row['manual_label']).strip() != '' else row['profile_type'],
        axis=1
    )
    
    thresholds = np.arange(0.1, 0.8, 0.05)
    f1_scores = []
    
    for thresh in thresholds:
        pred = labeled_with_conf.apply(
            lambda row: 'Default' if row['profile_type'] == 'Default' 
                        else ('Anime' if row['anime_conf'] >= thresh else 'Photo'),
            axis=1
        )
        f1 = f1_score(labeled_with_conf['manual_label'], pred, labels=labels, average='macro', zero_division=0)
        f1_scores.append(f1)
    
    best_idx = np.argmax(f1_scores)
    best_thresh = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]
    
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(thresholds, f1_scores, 'b-o', markersize=4)
    ax.axvline(x=best_thresh, color='red', linestyle='--', label=f'Best: conf={best_thresh:.2f} (F1={best_f1:.3f})')
    ax.axvline(x=0.3, color='gray', linestyle=':', alpha=0.5, label='Default: conf=0.3')
    ax.set_xlabel('Confidence Threshold')
    ax.set_ylabel('Macro F1-score')
    ax.set_title('F1-Score vs Confidence Threshold')
    ax.legend()
    plt.tight_layout()
    plt.savefig("../figures/conf_threshold_tuning.png", dpi=150)
    plt.show()
    
    print(f'Optimal threshold: {best_thresh:.2f} (Macro F1 = {best_f1:.3f})')
else:
    print('Run again after completing manual labels.')